# Samer's extra metrics on Colab: entities, sentences, E5 and sentence coverage

One Colab session for the extra metrics that would take over 10 minutes locally:
1. **spaCy pass** (CPU, ~40 min, uses all vCPUs): named entities, numbers, quantities and sentences for every text (`scripts/info_features.py`).
2. **Information recall** (CPU, a few minutes): entity / number / quantity recall (`scripts/score_extra.py information`).
3. **E5 cosine and sentence coverage** (GPU): `intfloat/e5-base-v2` document cosine and MPNet sentence coverage at cosine 0.6 (`scripts/score_neural_extra.py`, fp32).

chrF, word edit rate and exact equality take about 4 minutes and are already computed locally.

**Before running**
1. `MyDrive/sot_colab/sot_colab_bundle.zip` is already on Drive from the semantic run; it provides the corpus and the sample/floor files.
2. On the Mac, build the small code bundle and upload it to **`MyDrive/sot_colab/`**:
   ```bash
   zip -r -9 data/colab/sot_extra_bundle.zip src scripts -x '*__pycache__*'
   ```
3. **Runtime → Change runtime type → A100 GPU** (an A100 runtime also has the most vCPUs for step 1), then run all cells.

**Outputs** go to `MyDrive/sot_colab/derived/`. Download these four into the repo's `data/derived/`:
`sot_pairs_information_full.parquet`, `sot_floors_information.parquet`, `sot_pairs_neural_extra.parquet` and `sot_floors_neural_extra.parquet`.

**Disconnects:** every step saves progress to Drive. Reconnect and run all cells again; finished steps are skipped and unfinished ones resume.

In [ ]:
# 1. Drive, GPU, unpack (corpus from the semantic bundle, newest code from the extra bundle)
import os, shutil, subprocess, sys, time
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/sot_colab")
DERIVED, WORK = DRIVE / "derived", Path("/content/work")
for z in ("sot_colab_bundle.zip", "sot_extra_bundle.zip"):
    assert (DRIVE / z).exists(), f"upload {z} to {DRIVE} first"

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> A100 GPU"
GPU_MEM = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {torch.cuda.get_device_name(0)} ({GPU_MEM:.0f} GB) | vCPUs: {os.cpu_count()}")

if not (WORK / "data/sot/paraphrased_datasets").exists():
    shutil.unpack_archive(DRIVE / "sot_colab_bundle.zip", WORK)
shutil.unpack_archive(DRIVE / "sot_extra_bundle.zip", WORK)   # always: newest code overrides the old bundle's
DERIVED.mkdir(parents=True, exist_ok=True)
for name in ("sot_sample.parquet", "sot_floors.parquet"):
    if not (DERIVED / name).exists():
        shutil.copy2(WORK / "data/derived" / name, DERIVED / name)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz"], check=True)
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=True)

BATCH = {"jobs_per_batch": 32, "batch_size": 64} if GPU_MEM > 30 else \
        {"jobs_per_batch": 16, "batch_size": 32} if GPU_MEM > 20 else {"jobs_per_batch": 8, "batch_size": 16}
ENV = {**os.environ, "PYTHONPATH": str(WORK / "src"), "TOKENIZERS_PARALLELISM": "false"}


def run(script, *args, log):
    """Run a repo script against the Drive caches; stream progress lines and keep the full log on Drive."""
    cmd = [sys.executable, f"scripts/{script}", *args, "--derived", str(DERIVED)]
    print(" ".join(cmd[1:]), flush=True)
    t = time.time()
    with open(DERIVED / log, "a") as f, subprocess.Popen(
            cmd, cwd=WORK, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            if any(k in line for k in ("chunk", "jobs,", "done", "needed", "originals", "Error", "Traceback", "error")):
                print(line, end="", flush=True)
            f.write(line)
    if p.returncode:
        raise RuntimeError(f"{script} failed (exit {p.returncode}); see {DERIVED / log}")
    print(f"-> {script} finished in {(time.time() - t) / 60:.1f} min\n", flush=True)


print("ready; batch settings:", BATCH)

In [ ]:
# 2. spaCy pass + information recall (CPU), then E5 + sentence coverage (GPU). All resumable.
import pandas as pd

if not (DERIVED / "sot_sentences.parquet").exists():
    run("info_features.py", log="info_features.log")
if not (DERIVED / "sot_floors_information.parquet").exists():
    run("score_extra.py", "information", log="score_extra.log")
run("score_neural_extra.py", "--dtype", "fp32", "--jobs-per-batch", str(BATCH["jobs_per_batch"]),
    "--batch-size", str(BATCH["batch_size"]), log="neural_extra.log")

sample = pd.read_parquet(DERIVED / "sot_sample.parquet")
info_p = pd.read_parquet(DERIVED / "sot_pairs_information_full.parquet")
pairs_x = pd.read_parquet(DERIVED / "sot_pairs_neural_extra.parquet")
floors_x = pd.read_parquet(DERIVED / "sot_floors_neural_extra.parquet")
n_orig = pairs_x[["dataset", "source", "key"]].drop_duplicates().shape[0]
print(f"information: {len(info_p):,} pairs | neural extra: {len(pairs_x):,} rows for {n_orig:,} / {len(sample):,} "
      f"originals | floors: {len(floors_x):,} / 1,400")
assert len(info_p) == len(pairs_x) and n_orig == len(sample) and len(floors_x) == 1400, \
    "incomplete: run all cells again to resume"
print("Done. Download sot_pairs_information_full.parquet, sot_floors_information.parquet, "
      f"sot_pairs_neural_extra.parquet and sot_floors_neural_extra.parquet from {DERIVED} into the repo's data/derived/.")